[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-07-embeddings-vectorstores.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Embeddings and Vector Stores — Chroma and FAISS
**certified-journeys / llm-engineering-certified** · Day 7 · Retrieval Foundations

> **Goal for today:** Embed text with OpenAIEmbeddings, measure semantic similarity in NumPy, build a persistent Chroma vector store, and run similarity searches — the foundation of every RAG pipeline.

In [ ]:
%pip install -q langchain langchain-openai langchain-chroma chromadb faiss-cpu numpy openai

## Step 1 · What are embedding models?

An **embedding model** converts text into a dense numeric vector — a point in high-dimensional space — so that semantically similar texts end up close together.

| Property | Detail |
|---|---|
| Output | Fixed-length float vector (e.g. 1536 dims for `text-embedding-3-small`) |
| Similarity metric | Cosine similarity or dot product |
| Use case | Search, clustering, RAG retrieval, deduplication |
| LangChain base class | `Embeddings` — `.embed_documents(texts)`, `.embed_query(text)` |

The LangChain [embedding models concept page](https://python.langchain.com/docs/concepts/embedding_models/) covers every built-in provider.

In [ ]:
import os
import numpy as np
from langchain_openai import OpenAIEmbeddings

# Set your API key — in Colab use: os.environ['OPENAI_API_KEY'] = 'sk-...'
# Or add it as a Colab secret named OPENAI_API_KEY
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536-dim, cheapest OpenAI model

sentences = [
    "The Eiffel Tower is located in Paris.",
    "Paris is the capital of France.",
    "The Great Wall of China stretches thousands of miles.",
    "Machine learning models learn from data.",
    "Deep learning is a subset of machine learning.",
]

# embed_documents returns List[List[float]]
vectors = embed_model.embed_documents(sentences)
print(f"Embedded {len(vectors)} sentences")
print(f"Vector dimensionality: {len(vectors[0])}")

### What just happened?
- `embed_documents` sent all 5 sentences to the OpenAI embedding endpoint in one batched call.
- Each sentence returned as a **1536-dimensional float vector**.
- **Vectors are the raw currency** of all retrieval — downstream similarity search operates entirely on these numbers, never on the original text.

## Step 2 · Pairwise cosine similarity in NumPy

Cosine similarity measures the angle between two vectors, ranging from **-1** (opposite) to **1** (identical direction).  
It ignores vector magnitude, making it robust to text length differences.

$$\text{cosine}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

Computing the full pairwise matrix lets us see which sentence pairs the model considers semantically close — a useful sanity check before building a retrieval system.

In [ ]:
def cosine_similarity_matrix(vecs):
    """Compute N×N pairwise cosine similarity matrix."""
    arr = np.array(vecs)                     # shape (N, D)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)  # shape (N, 1)
    normed = arr / (norms + 1e-10)           # unit vectors
    return normed @ normed.T                 # dot product of unit vectors = cosine sim

sim_matrix = cosine_similarity_matrix(vectors)

print("Pairwise cosine similarity matrix (rounded to 3 dp):")
labels = [f"S{i}" for i in range(len(sentences))]
header = "      " + "  ".join(labels)
print(header)
for i, row in enumerate(sim_matrix):
    row_str = "  ".join(f"{v:.3f}" for v in row)
    print(f"{labels[i]}  {row_str}")

print("\nSentence legend:")
for i, s in enumerate(sentences):
    print(f"  S{i}: {s}")

### What just happened?
- **S0 ↔ S1** (both about Paris) should show high similarity (~0.85+).
- **S3 ↔ S4** (ML/deep learning) should also score high.
- **S0 ↔ S3** (Paris vs ML) should score much lower — the model has learned semantic distance.
- **Diagonal is always 1.0** — a sentence is perfectly similar to itself.
- This matrix is exactly what a vector store computes at search time, but over millions of stored vectors.

## Step 3 · Build a Chroma vector store

[Chroma](https://python.langchain.com/docs/integrations/vectorstores/chroma/) is the most popular **local-first** vector database for LangChain development.

| Feature | Chroma | FAISS | pgvector |
|---|---|---|---|
| Persistence | Yes (disk) | Manual | Yes (Postgres) |
| Multi-user | No | No | Yes |
| Metadata filter | Yes | Limited | Yes |
| Best for | Local dev & demos | In-memory search | Production |

`from_documents()` embeds every `Document` and stores the vectors in one call.

In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Create Document objects — metadata can hold source, page, date, etc.
docs = [
    Document(page_content="The Eiffel Tower is located in Paris.",
             metadata={"topic": "landmarks", "country": "France"}),
    Document(page_content="Paris is the capital of France.",
             metadata={"topic": "geography", "country": "France"}),
    Document(page_content="The Great Wall of China stretches thousands of miles.",
             metadata={"topic": "landmarks", "country": "China"}),
    Document(page_content="Machine learning models learn from data.",
             metadata={"topic": "AI"}),
    Document(page_content="Deep learning is a subset of machine learning.",
             metadata={"topic": "AI"}),
    Document(page_content="Neural networks are inspired by the human brain.",
             metadata={"topic": "AI"}),
    Document(page_content="The Louvre museum is in Paris and houses the Mona Lisa.",
             metadata={"topic": "landmarks", "country": "France"}),
    Document(page_content="Python is the most popular language for data science.",
             metadata={"topic": "programming"}),
]

PERSIST_DIR = "/tmp/chroma_day07"  # in Colab /tmp is writable

# from_documents embeds each doc and writes to the persist directory
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embed_model,
    persist_directory=PERSIST_DIR,
    collection_name="day07_demo",
)

print(f"Vector store created with {vectorstore._collection.count()} documents")
print(f"Persisted to: {PERSIST_DIR}")

### What just happened?
- `from_documents()` called the embedding model once per document (or in batches) and stored `(vector, metadata, page_content)` tuples in Chroma.
- **`persist_directory`** tells Chroma to write to disk — without it the store lives only in memory for the current session.
- The `collection_name` lets you store multiple independent corpora in the same directory.

## Step 4 · Similarity search and scored search

Two main retrieval methods:

| Method | Returns | When to use |
|---|---|---|
| `similarity_search(query, k)` | `List[Document]` | Default retrieval — clean API |
| `similarity_search_with_score(query, k)` | `List[(Document, float)]` | Debug relevance, set score thresholds |

The score is **L2 distance** in Chroma's default config (lower = more similar). Some stores return cosine similarity (higher = more similar) — always check the docs.

In [ ]:
query = "famous tourist attractions in France"

# --- Basic similarity search ---
print("=" * 60)
print(f"Query: {query!r}")
print("\n--- similarity_search (top 3) ---")
results = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(results, 1):
    print(f"  [{i}] {doc.page_content}")
    print(f"       metadata: {doc.metadata}")

# --- Scored search ---
print("\n--- similarity_search_with_score (top 5) ---")
scored = vectorstore.similarity_search_with_score(query, k=5)
for doc, score in scored:
    print(f"  score={score:.4f}  |  {doc.page_content}")

print("\nNote: Chroma returns L2 distance — LOWER score = MORE relevant")

### What just happened?
- Chroma embedded the query on-the-fly, then found the `k` nearest stored vectors using L2 distance.
- **France-related landmarks ranked highest** — the model correctly associated "tourist attractions in France" with Paris and the Eiffel Tower.
- The scored search reveals the **score gap** between highly relevant and weakly relevant results — useful for deciding a retrieval threshold.
- **Metadata comes back intact** — you can filter post-retrieval or pass it downstream as citation data.

## Step 5 · Persist to disk and reload without re-embedding

Re-embedding at startup wastes time and API credits. Chroma lets you save the store once and reload it later with just the embedding model (for query-time embedding only).

**Pattern:** embed once → persist → reload → query forever.

In [ ]:
import os

# Verify files were written to disk
print("Files in persist directory:")
for root, dirs, files in os.walk(PERSIST_DIR):
    for f in files:
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        print(f"  {full}  ({size_kb:.1f} KB)")

# --- Reload from disk (no documents passed, no re-embedding) ---
reloaded = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embed_model,  # needed for query-time embedding only
    collection_name="day07_demo",
)

count = reloaded._collection.count()
print(f"\nReloaded store has {count} documents (no re-embedding performed)")

# Sanity-check: same query should return same results
reload_results = reloaded.similarity_search("machine learning and AI", k=2)
print("\nQuery: 'machine learning and AI' (top 2 from reloaded store):")
for doc in reload_results:
    print(f"  • {doc.page_content}")

### What just happened?
- Chroma stores vectors as SQLite + binary files on disk — reloading reads those files directly, skipping all embedding API calls.
- The `embedding_function` is still needed at reload time to **embed new queries** at search time.
- **This pattern is production-ready for small-to-medium corpora** — build the index offline, ship the files, reload at runtime.

## Step 6 · Bonus — FAISS in-memory vector store

FAISS (Facebook AI Similarity Search) is a pure in-memory alternative — faster for large corpora but requires manual save/load. Good for prototyping pipelines where you don't need persistence across sessions.

In [ ]:
from langchain_community.vectorstores import FAISS

# FAISS.from_documents works the same interface as Chroma
faiss_store = FAISS.from_documents(docs, embed_model)

faiss_results = faiss_store.similarity_search_with_score("AI and neural networks", k=3)
print("FAISS results for 'AI and neural networks':")
for doc, score in faiss_results:
    # FAISS returns L2 distance by default (lower = better)
    print(f"  score={score:.4f}  |  {doc.page_content}")

# Save and reload FAISS index
FAISS_DIR = "/tmp/faiss_day07"
faiss_store.save_local(FAISS_DIR)
reloaded_faiss = FAISS.load_local(
    FAISS_DIR,
    embed_model,
    allow_dangerous_deserialization=True,  # required flag for local loads
)
print(f"\nFAISS reloaded from {FAISS_DIR}")
print(f"Index contains {reloaded_faiss.index.ntotal} vectors")

### What just happened?
- FAISS stores vectors in RAM as a flat index — blazing fast for < 1M vectors.
- `save_local` / `load_local` serialize to two binary files: the index and the docstore.
- **`allow_dangerous_deserialization=True`** is required because FAISS uses pickle under the hood — only load indexes you created yourself.
- For production at scale, replace FAISS with **pgvector** (Postgres) or **Pinecone** — the LangChain interface stays identical.

In [ ]:
# Challenge: Build a topic-filtered vector store
#
# 1. Add 5 new Documents to the existing Chroma store — pick any topic
#    and include meaningful metadata (e.g. {"topic": "history", "year": 2024})
# 2. Run similarity_search_with_score for a query relevant to your new docs
# 3. Filter results to only return docs where metadata["topic"] == your chosen topic
#    Hint: Chroma supports filter= kwarg:  vectorstore.similarity_search(q, k=5, filter={"topic": "..."})
# 4. Compare scored results WITH and WITHOUT the metadata filter
#
# Your solution here:
new_docs = [
    # Document(page_content="...", metadata={"topic": "..."}),
]

# vectorstore.add_documents(new_docs)

# challenge_query = "..."
# unfiltered = vectorstore.similarity_search_with_score(challenge_query, k=5)
# filtered   = vectorstore.similarity_search_with_score(challenge_query, k=5, filter={"topic": "..."})

---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Embedding model | Maps text → dense float vector; similar texts land close in vector space |
| Cosine similarity | Angle between vectors; 1 = identical direction, 0 = orthogonal |
| `embed_documents` vs `embed_query` | Batch vs single; some models use separate encoders for each |
| Chroma `from_documents` | One-shot embed + store; use `persist_directory` to save to disk |
| `similarity_search_with_score` | Returns (doc, distance) — use scores to debug retrieval quality |
| FAISS | In-memory alternative; faster for large corpora, no built-in persistence |
| Metadata filters | Narrow retrieval to a subset of docs without re-embedding |

> **Tip:** Chroma is the easiest persistent store for local development. Switch to pgvector or Pinecone only when you need multi-user access or cloud scalability.

---
## What's next
**Day 8** → Retrieval-Augmented Generation — wire your Chroma store into a full RAG chain using LCEL, add MMR retrieval, and attach source citations to every answer.

Mark Day 7 complete in your [tracker](../index.html).